# 🔮 Interpretabilidad de Modelos: SHAP y Feature Importance

**Módulo 03** | **Sesión 7** | **Duración estimada: 1h 30min**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gonzalezulises/formacion-docente-bi-faces/blob/main/modulo-03-modelado-predictivo/notebooks/03_07_interpretabilidad_shap.ipynb)

## 🎯 Objetivos de Aprendizaje

Al finalizar esta sesión, serás capaz de:

| # | Competencia | Nivel |
|---|-------------|-------|
| 1 | Explicar por qué la interpretabilidad es crucial al comunicar resultados de modelos predictivos | Comprensión |
| 2 | Calcular y visualizar la importancia de variables (Feature Importance) de modelos basados en árboles | Aplicación |
| 3 | Comparar Feature Importance nativa vs. Permutation Importance y entender sus diferencias | Análisis |
| 4 | Aplicar SHAP para explicar predicciones de forma global (¿qué variables son más importantes?) y local (¿por qué este resultado específico?) | Aplicación |
| 5 | Interpretar gráficos SHAP (summary, bar, waterfall) y traducirlos a recomendaciones de política | Síntesis |
| 6 | Comunicar resultados de modelos a audiencias no técnicas de manera efectiva | Evaluación |

## 💡 ¿Por qué es importante para tu práctica docente?

Imagina que entrenas un modelo que predice qué estudiantes están en riesgo de deserción. El modelo funciona bien (80% de precisión), pero el comité académico te pregunta: **"¿Por qué el modelo dice que este estudiante va a desertar?"**

Si no puedes responder, el modelo es una **caja negra** inútil para la toma de decisiones.

En FACES-UC, nuestros resultados se presentan ante:
- **Consejos de Facultad** que necesitan entender la lógica detrás de las recomendaciones
- **Estudiantes** que merecen explicaciones transparentes sobre su situación académica
- **Comités de investigación** que evalúan la rigurosidad metodológica
- **Tomadores de decisión institucional** que traducen datos en políticas

La interpretabilidad NO es un lujo técnico — es una **responsabilidad ética y profesional.**

> **En esta sesión:** Aprenderás a abrir la caja negra de tus modelos usando herramientas que te permiten explicar **cada predicción** en lenguaje claro.

In [ ]:
# ============================================================
# Configuración inicial - Ejecutar esta celda primero
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, r2_score, classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder
import shap
import warnings

# Suprimir advertencias que no son relevantes para el aprendizaje
warnings.filterwarnings('ignore')

# Inicializar SHAP para visualización en notebooks
shap.initjs()

# Configuración visual
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

# Semilla para reproducibilidad
np.random.seed(42)

print('✅ Bibliotecas cargadas correctamente')
print(f'   pandas {pd.__version__}')
print(f'   shap {shap.__version__}')

---

## 🔲 1. El problema de la caja negra

### ¿Por qué los modelos de ML son difíciles de explicar?

Los modelos de Machine Learning pueden ser muy precisos, pero su complejidad interna los hace opacos:

| Modelo | Interpretabilidad | Precisión típica | ¿Por qué? |
|--------|:-----------------:|:----------------:|------------|
| Regresión lineal | ⭐⭐⭐⭐⭐ | ⭐⭐ | Cada coeficiente tiene significado claro |
| Árbol de decisión | ⭐⭐⭐⭐ | ⭐⭐⭐ | Se puede visualizar cada regla |
| Random Forest | ⭐⭐ | ⭐⭐⭐⭐ | Promedio de 100+ árboles — ¿cuál domina? |
| Gradient Boosting | ⭐⭐ | ⭐⭐⭐⭐⭐ | Cadena de correcciones sucesivas |
| Red neuronal | ⭐ | ⭐⭐⭐⭐⭐ | Millones de pesos interconectados |

### El trade-off entre precisión e interpretabilidad

Generalmente, los modelos más precisos son los más difíciles de explicar. Pero **no tenemos que elegir:**

**SHAP** y otras herramientas de interpretabilidad nos permiten tomar un modelo complejo (como Random Forest o Gradient Boosting) y **explicar sus predicciones como si fuera un modelo simple.**

### ¿Por qué no simplemente usar modelos simples?

Porque en problemas reales (predecir deserción, evaluar riesgo financiero, analizar rendimiento), los modelos complejos capturan **relaciones no lineales e interacciones** que los modelos simples no pueden ver. La solución no es renunciar a la precisión, sino **agregar una capa de explicación.**

---

## 📊 2. Feature Importance (Importancia de Variables)

La forma más directa de entender un modelo basado en árboles es preguntarle: **"¿Qué variables usaste más para tomar decisiones?"**

Los modelos como Random Forest tienen una métrica incorporada llamada **feature importance** que mide cuánto contribuye cada variable a reducir el error de predicción.

In [ ]:
# ============================================================
# Cargar datos de rendimiento académico
# ============================================================
estudiantes = pd.read_csv('../../datasets/universidad/rendimiento_academico.csv')

print('=== Dataset: Rendimiento Académico ===')
print(f'Estudiantes: {len(estudiantes):,}')
print(f'Variables: {list(estudiantes.columns)}')
print(f'\nEstadísticas del promedio (variable objetivo):')
print(f'   Media:  {estudiantes["promedio"].mean():.1f}')
print(f'   Mín:    {estudiantes["promedio"].min():.1f}')
print(f'   Máx:    {estudiantes["promedio"].max():.1f}')
print(f'\nPrimeras filas:')
display(estudiantes.head())

In [ ]:
# ============================================================
# Preparar datos para el modelo
# ============================================================

# Variables predictoras (features)
features = ['semestre', 'edad', 'asistencia_pct', 'materias_inscritas', 'beca', 'trabaja']

# Convertir booleanos a numéricos (True/False → 1/0)
estudiantes['beca'] = estudiantes['beca'].astype(int)
estudiantes['trabaja'] = estudiantes['trabaja'].astype(int)

# Separar en X (predictores) e y (objetivo)
X = estudiantes[features].copy()
y = estudiantes['promedio'].copy()

# Dividir en entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Datos de entrenamiento: {len(X_train):,} estudiantes')
print(f'Datos de prueba:        {len(X_test):,} estudiantes')
print(f'\nVariables predictoras:')
for f in features:
    print(f'   • {f}: {X[f].dtype} — rango [{X[f].min()}, {X[f].max()}]')

In [ ]:
# ============================================================
# Entrenar Random Forest y obtener Feature Importance
# ============================================================

# Entrenar el modelo
rf_modelo = RandomForestRegressor(
    n_estimators=200,    # 200 árboles
    max_depth=10,        # Profundidad máxima
    random_state=42
)
rf_modelo.fit(X_train, y_train)

# Evaluar el modelo
y_pred = rf_modelo.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print('=== Rendimiento del modelo ===')
print(f'   Error Absoluto Medio (MAE): {mae:.2f} puntos')
print(f'   R² Score: {r2:.3f}')
print(f'\n   Interpretación: El modelo se equivoca en promedio {mae:.1f} puntos')
print(f'   al predecir la nota de un estudiante, y explica el {r2*100:.1f}%')
print(f'   de la variabilidad en el promedio.')

In [ ]:
# ============================================================
# Feature Importance: ¿Qué factores afectan más el promedio?
# ============================================================

# Obtener importancias del modelo
importancias = pd.DataFrame({
    'Variable': features,
    'Importancia': rf_modelo.feature_importances_
}).sort_values('Importancia', ascending=True)

# Graficar como barras horizontales
fig, ax = plt.subplots(figsize=(10, 5))

colores = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(importancias)))
ax.barh(importancias['Variable'], importancias['Importancia'], color=colores, edgecolor='white')

# Agregar porcentajes
for i, (_, row) in enumerate(importancias.iterrows()):
    ax.text(row['Importancia'] + 0.005, i, f"{row['Importancia']*100:.1f}%", 
            va='center', fontweight='bold', fontsize=11)

ax.set_title('¿Qué factores afectan más el promedio académico?', fontsize=14, fontweight='bold')
ax.set_xlabel('Importancia relativa')
ax.set_xlim(0, importancias['Importancia'].max() * 1.15)

plt.tight_layout()
plt.show()

# Interpretación
top_var = importancias.iloc[-1]
print(f'\n🔍 Interpretación:')
print(f'   La variable más importante es "{top_var["Variable"]}" ')
print(f'   ({top_var["Importancia"]*100:.1f}% de la importancia total).')
print(f'\n   Esto significa que el modelo usa esta variable más que cualquier otra')
print(f'   para distinguir entre estudiantes con alto y bajo promedio.')
print(f'\n   ⚠️ OJO: Importancia NO significa causalidad.')
print(f'   Que "asistencia" sea importante no prueba que asistir CAUSE mejores notas.')
print(f'   Pero sí indica una fuerte asociación que merece investigarse.')

---

## 🔀 3. Permutation Importance

La Feature Importance nativa de Random Forest puede ser **engañosa** en algunos casos:
- Tiende a favorecer variables con muchos valores posibles (como `edad` o `semestre`)
- Puede inflar la importancia de variables correlacionadas

**Permutation Importance** ofrece una alternativa más robusta. La idea es simple:

1. Medir el rendimiento del modelo con los datos originales
2. Para cada variable, **desordenar aleatoriamente** sus valores (permutarla)
3. Medir cuánto **empeoró** el modelo
4. Si empeora mucho → esa variable es muy importante
5. Si empeora poco → el modelo no la necesita realmente

Es como evaluar a un equipo de fútbol reemplazando un jugador por alguien aleatorio: si el equipo juega igual, ese jugador no era tan importante.

In [ ]:
# ============================================================
# Permutation Importance
# ============================================================

# Calcular Permutation Importance sobre datos de prueba
perm_importance = permutation_importance(
    rf_modelo, X_test, y_test, 
    n_repeats=30,       # Repetir 30 veces para estabilidad
    random_state=42
)

# Crear DataFrame con resultados
perm_df = pd.DataFrame({
    'Variable': features,
    'Importancia_media': perm_importance.importances_mean,
    'Importancia_std': perm_importance.importances_std
}).sort_values('Importancia_media', ascending=True)

# Comparar ambos métodos lado a lado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel izquierdo: Feature Importance nativa
imp_nativa = importancias.sort_values('Importancia', ascending=True)
axes[0].barh(imp_nativa['Variable'], imp_nativa['Importancia'], 
             color='#3498db', edgecolor='white')
axes[0].set_title('Feature Importance\n(nativa del modelo)', fontweight='bold')
axes[0].set_xlabel('Importancia')

# Panel derecho: Permutation Importance
axes[1].barh(perm_df['Variable'], perm_df['Importancia_media'], 
             xerr=perm_df['Importancia_std'], color='#e67e22', 
             edgecolor='white', capsize=3)
axes[1].set_title('Permutation Importance\n(basada en rendimiento)', fontweight='bold')
axes[1].set_xlabel('Caída en R² al permutar')

plt.suptitle('Comparación de métodos de importancia de variables', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('🔍 Compara ambos gráficos:')
print('   • ¿El ranking es similar o cambia?')
print('   • Las barras de error (Permutation) indican variabilidad — barras grandes = menos confiable')
print('   • Si ambos métodos coinciden, la conclusión es más robusta')

---

## 🧩 4. Introducción a SHAP

### ¿Qué son los valores SHAP?

Imagina que un grupo de 6 personas (nuestras 6 variables) trabajan juntas en un proyecto y obtienen una nota. La pregunta es: **¿Cuánto contribuyó cada persona al resultado?**

Los **valores de Shapley** (de la teoría de juegos) resuelven este problema de forma justa:

1. Se evalúa el resultado con **todas las combinaciones posibles** de jugadores
2. Se calcula la **contribución marginal** de cada jugador en cada combinación
3. Se promedian todas las contribuciones marginales

### En Machine Learning:

- Los "jugadores" son las **variables** del modelo (edad, asistencia, beca...)
- El "resultado" es la **predicción** del modelo
- SHAP nos dice cuánto **sube o baja** la predicción por culpa de cada variable

### ¿Por qué SHAP es mejor que Feature Importance?

| Feature Importance | SHAP |
|-------------------|------|
| Solo dice qué variable es importante | Dice **cuánto y en qué dirección** afecta |
| Solo da importancia **global** | Da explicaciones **globales Y locales** (por estudiante) |
| No distingue efecto positivo vs negativo | Muestra si la variable **sube o baja** la predicción |

En resumen: Feature Importance dice **"la asistencia importa"**. SHAP dice **"para este estudiante, su baja asistencia (65%) redujo la predicción de promedio en 1.8 puntos".**

---

## 🔬 5. SHAP en la práctica

Vamos a aplicar SHAP a nuestro modelo de predicción de promedio académico paso a paso.

In [ ]:
# ============================================================
# Crear el explicador SHAP
# ============================================================

# TreeExplainer es el método optimizado para modelos basados en árboles
# (Random Forest, Gradient Boosting, XGBoost, LightGBM)
explicador = shap.TreeExplainer(rf_modelo)

# Calcular valores SHAP para los datos de prueba
# Esto puede tomar unos segundos...
print('Calculando valores SHAP para', len(X_test), 'estudiantes...')
shap_values = explicador(X_test)

print(f'✅ Valores SHAP calculados')
print(f'   Dimensiones: {shap_values.values.shape}')
print(f'   → {shap_values.values.shape[0]} estudiantes × {shap_values.values.shape[1]} variables')
print(f'   Cada celda dice cuánto cambia la predicción por esa variable para ese estudiante.')

In [ ]:
# ============================================================
# Gráfico 1: Summary Plot (Beeswarm) — Visión global
# ============================================================

print('📊 SUMMARY PLOT (Beeswarm)')
print('=' * 50)
print('Cómo leerlo:')
print('  • Cada punto = 1 estudiante')
print('  • Eje horizontal = efecto SHAP (cuánto cambia la predicción)')
print('  • Color = valor de la variable (rojo = alto, azul = bajo)')
print('  • Variables ordenadas de arriba a abajo por importancia')
print()

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, show=False)
plt.title('SHAP: ¿Cómo afecta cada variable al promedio predicho?', 
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print('\n🔍 Interpretación del gráfico:')
print('   • Si una variable tiene puntos ROJOS a la DERECHA: valores ALTOS de esa variable')
print('     AUMENTAN el promedio predicho.')
print('   • Si una variable tiene puntos AZULES a la DERECHA: valores BAJOS de esa variable')
print('     AUMENTAN el promedio predicho (relación inversa).')
print('   • Variables con puntos más dispersos horizontalmente tienen más impacto.')

In [ ]:
# ============================================================
# Gráfico 2: Bar Plot — Importancia global según SHAP
# ============================================================

print('📊 BAR PLOT — Importancia global según SHAP')
print('=' * 50)
print('Muestra la magnitud promedio del efecto de cada variable.')
print()

plt.figure(figsize=(10, 5))
shap.plots.bar(shap_values, show=False)
plt.title('SHAP: Importancia global de cada variable', 
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Gráfico 3: Waterfall — Explicar UNA predicción individual
# ============================================================

# Seleccionar un estudiante específico del conjunto de prueba
idx = 0  # Primer estudiante del conjunto de prueba

# Obtener la predicción y datos reales
pred_individual = rf_modelo.predict(X_test.iloc[[idx]])[0]
real_individual = y_test.iloc[idx]
datos_estudiante = X_test.iloc[idx]

print('📊 WATERFALL PLOT — Explicación de una predicción individual')
print('=' * 60)
print(f'\n👤 Estudiante seleccionado:')
for var in features:
    print(f'   • {var}: {datos_estudiante[var]}')
print(f'\n   Promedio REAL:     {real_individual:.1f}')
print(f'   Promedio PREDICHO: {pred_individual:.1f}')
print(f'\n¿Por qué el modelo predice {pred_individual:.1f}?')
print('El waterfall muestra la contribución de cada variable:')
print()

plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values[idx], show=False)
plt.title(f'¿Por qué el modelo predice un promedio de {pred_individual:.1f}?', 
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print('\n🔍 Cómo leer el waterfall:')
print(f'   • La base (E[f(x)]) es el promedio general de predicciones: {shap_values.base_values[idx]:.1f}')
print(f'   • Cada barra muestra cuánto SUBE (rojo) o BAJA (azul) la predicción')
print(f'   • El resultado final es la predicción: f(x) = {pred_individual:.1f}')

In [ ]:
# ============================================================
# Veamos otro estudiante con características diferentes
# ============================================================

# Buscar un estudiante con predicción alta
predicciones_test = rf_modelo.predict(X_test)
idx_alto = np.argmax(predicciones_test)

pred_alto = predicciones_test[idx_alto]
real_alto = y_test.iloc[idx_alto]
datos_alto = X_test.iloc[idx_alto]

print(f'👤 Estudiante con MAYOR promedio predicho:')
for var in features:
    print(f'   • {var}: {datos_alto[var]}')
print(f'\n   Promedio REAL:     {real_alto:.1f}')
print(f'   Promedio PREDICHO: {pred_alto:.1f}')
print()

plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values[idx_alto], show=False)
plt.title(f'Estudiante destacado: ¿Por qué el modelo predice {pred_alto:.1f}?', 
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

---

## 🎓 6. Caso aplicado: Deserción estudiantil

Ahora aplicaremos todo lo aprendido a un problema real y urgente para FACES-UC: **identificar estudiantes en riesgo de deserción** y entender **qué factores los ponen en riesgo.**

### Definición operativa:
- Estudiante **"en riesgo"** = promedio menor a 10 (escala 1-20)
- Estudiante **"no en riesgo"** = promedio 10 o mayor

Este es un **problema de clasificación** (en riesgo / no en riesgo), así que usaremos **Gradient Boosting Classifier.**

In [ ]:
# ============================================================
# Crear variable objetivo: en riesgo de deserción
# ============================================================

# Definir: promedio < 10 = en riesgo
estudiantes['en_riesgo'] = (estudiantes['promedio'] < 10).astype(int)

# Ver distribución
conteo = estudiantes['en_riesgo'].value_counts()
print('=== Distribución de riesgo de deserción ===')
print(f'   No en riesgo (promedio >= 10): {conteo[0]:,} estudiantes ({conteo[0]/len(estudiantes)*100:.1f}%)')
print(f'   En riesgo (promedio < 10):     {conteo[1]:,} estudiantes ({conteo[1]/len(estudiantes)*100:.1f}%)')

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histograma de promedios con línea de corte
axes[0].hist(estudiantes['promedio'], bins=30, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].axvline(x=10, color='red', linestyle='--', linewidth=2, label='Umbral: 10')
axes[0].set_title('Distribución de promedios', fontweight='bold')
axes[0].set_xlabel('Promedio')
axes[0].legend()

# Pie chart
axes[1].pie([conteo[0], conteo[1]], labels=['No en riesgo', 'En riesgo'], 
            colors=['#27ae60', '#e74c3c'], autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 12})
axes[1].set_title('Proporción de estudiantes en riesgo', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Entrenar modelo de clasificación (Gradient Boosting)
# ============================================================

# Preparar datos
X_clf = estudiantes[features].copy()
y_clf = estudiantes['en_riesgo'].copy()

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

# Entrenar Gradient Boosting
gb_modelo = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)
gb_modelo.fit(X_train_c, y_train_c)

# Evaluar
y_pred_c = gb_modelo.predict(X_test_c)
acc = accuracy_score(y_test_c, y_pred_c)

print('=== Modelo de Riesgo de Deserción ===')
print(f'   Precisión general: {acc*100:.1f}%')
print(f'\nReporte detallado:')
print(classification_report(y_test_c, y_pred_c, 
                           target_names=['No en riesgo', 'En riesgo']))

In [ ]:
# ============================================================
# Análisis SHAP del modelo de deserción
# ============================================================

# Crear explicador SHAP para el modelo de clasificación
explicador_clf = shap.TreeExplainer(gb_modelo)
shap_values_clf = explicador_clf(X_test_c)

print('📊 ¿Qué factores ponen a los estudiantes en riesgo de deserción?')
print('=' * 60)
print()

# Summary plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_clf, X_test_c, show=False)
plt.title('SHAP: Factores de riesgo de deserción estudiantil', 
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print('\n🔍 Lectura del gráfico (recordar: predicción = 1 es "en riesgo"):')
print('   • Puntos ROJOS a la DERECHA: valores ALTOS de esa variable AUMENTAN el riesgo')
print('   • Puntos AZULES a la DERECHA: valores BAJOS de esa variable AUMENTAN el riesgo')

In [ ]:
# ============================================================
# Comparar 2 estudiantes: uno en riesgo y otro no
# ============================================================

# Encontrar un estudiante en riesgo y uno no en riesgo
idx_riesgo = y_test_c[y_test_c == 1].index[0]
idx_no_riesgo = y_test_c[y_test_c == 0].index[0]

# Posiciones en X_test_c
pos_riesgo = X_test_c.index.get_loc(idx_riesgo)
pos_no_riesgo = X_test_c.index.get_loc(idx_no_riesgo)

# --- Estudiante EN RIESGO ---
datos_r = X_test_c.iloc[pos_riesgo]
prob_r = gb_modelo.predict_proba(X_test_c.iloc[[pos_riesgo]])[0]

print('👤 ESTUDIANTE EN RIESGO')
print('-' * 40)
for var in features:
    print(f'   {var}: {datos_r[var]}')
print(f'   Probabilidad de riesgo: {prob_r[1]*100:.1f}%')
print()

plt.figure(figsize=(10, 5))
shap.plots.waterfall(shap_values_clf[pos_riesgo], show=False)
plt.title('¿Por qué este estudiante está en riesgo?', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# --- Estudiante NO EN RIESGO ---
datos_nr = X_test_c.iloc[pos_no_riesgo]
prob_nr = gb_modelo.predict_proba(X_test_c.iloc[[pos_no_riesgo]])[0]

print('👤 ESTUDIANTE NO EN RIESGO')
print('-' * 40)
for var in features:
    print(f'   {var}: {datos_nr[var]}')
print(f'   Probabilidad de riesgo: {prob_nr[1]*100:.1f}%')
print()

plt.figure(figsize=(10, 5))
shap.plots.waterfall(shap_values_clf[pos_no_riesgo], show=False)
plt.title('¿Por qué este estudiante NO está en riesgo?', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print('\n💡 Implicaciones para política académica:')
print('   Al comparar ambos estudiantes, podemos identificar los factores')
print('   que más distinguen a estudiantes en riesgo de los que no lo están.')
print('   Esto permite diseñar intervenciones FOCALIZADAS:')
print('   - Si "asistencia" es clave → programas de seguimiento a inasistencia')
print('   - Si "trabaja" es clave → horarios flexibles o becas de dedicación')
print('   - Si "materias_inscritas" es clave → limitar carga académica en riesgo')

---

## 💰 7. SHAP para modelos de regresión con datos económicos

SHAP no es solo para datos académicos. Veamos un ejemplo rápido con **datos financieros empresariales:** predecir la utilidad neta a partir de variables contables.

In [ ]:
# ============================================================
# Cargar datos de estados financieros
# ============================================================

financieros = pd.read_csv('../../datasets/empresarial/estados_financieros.csv')

print('=== Dataset: Estados Financieros ===')
print(f'Registros: {len(financieros)} (empresa × año)')
print(f'Columnas: {list(financieros.columns)}')
display(financieros.head())

In [ ]:
# ============================================================
# Modelo: Predecir utilidad neta
# ============================================================

# Variables predictoras
features_fin = ['activos_musd', 'pasivos_musd', 'ingresos_musd', 'costos_musd']

X_fin = financieros[features_fin].copy()
y_fin = financieros['utilidad_neta_musd'].copy()

# Entrenar Random Forest
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fin, y_fin, test_size=0.2, random_state=42
)

rf_fin = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
rf_fin.fit(X_train_f, y_train_f)

# Evaluar
r2_fin = r2_score(y_test_f, rf_fin.predict(X_test_f))
mae_fin = mean_absolute_error(y_test_f, rf_fin.predict(X_test_f))

print(f'Modelo de utilidad neta:')
print(f'   R²: {r2_fin:.3f}')
print(f'   MAE: ${mae_fin:,.0f} miles USD')

# SHAP
explicador_fin = shap.TreeExplainer(rf_fin)
shap_values_fin = explicador_fin(X_test_f)

# Summary plot
plt.figure(figsize=(10, 5))
shap.summary_plot(shap_values_fin, X_test_f, show=False)
plt.title('SHAP: ¿Qué determina la utilidad neta de una empresa?', 
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SHAP Dependence Plot: Relación activos → utilidad
# ============================================================

print('📊 DEPENDENCE PLOT: Activos vs. Utilidad Neta predicha')
print('=' * 55)
print('Muestra cómo cambia el efecto SHAP de una variable según su valor.')
print('Los colores indican la interacción con otra variable.')
print()

plt.figure(figsize=(10, 6))
shap.plots.scatter(shap_values_fin[:, 'activos_musd'], color=shap_values_fin[:, 'ingresos_musd'], show=False)
plt.title('Efecto de los activos sobre la utilidad neta predicha', 
          fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Activos (miles USD)')
plt.ylabel('Efecto SHAP sobre utilidad neta')
plt.tight_layout()
plt.show()

print('\n🔍 Interpretación:')
print('   • Eje X: valor de los activos de cada empresa')
print('   • Eje Y: cuánto contribuyen los activos a la predicción de utilidad')
print('   • Color: nivel de ingresos (rojo=alto, azul=bajo)')
print('   • Si la tendencia es ascendente: más activos → mayor utilidad predicha')
print('   • Pero el color puede mostrar que el efecto depende del nivel de ingresos')

---

## 🗣️ 8. Comunicando resultados interpretables

Tener los gráficos SHAP es solo la mitad del trabajo. La otra mitad es **comunicarlos efectivamente** a audiencias no técnicas.

### Consejos prácticos:

| En vez de decir... | Di... |
|---------------------|-------|
| "El valor SHAP de asistencia es +2.3" | "La alta asistencia de este estudiante sube su promedio predicho en 2.3 puntos" |
| "El feature importance de costos es 0.35" | "Los costos son el factor más determinante de la utilidad, explicando el 35% del modelo" |
| "El p-valor del ADF es 0.02" | "Los datos muestran un patrón estable que permite hacer pronósticos confiables" |

### Plantilla para presentar resultados SHAP:

1. **Contexto:** "Entrenamos un modelo para predecir [variable objetivo] usando [N] variables"
2. **Rendimiento:** "El modelo acierta en el [X]% de los casos"
3. **Hallazgo global:** "Los factores más importantes son [1, 2, 3]"
4. **Ejemplo individual:** "Para el caso de [persona/empresa], la predicción se explica porque..."
5. **Recomendación:** "Esto sugiere que deberíamos enfocarnos en..."

### Ejemplo para el caso de deserción:

> *"Nuestro modelo identifica correctamente al 85% de los estudiantes en riesgo de deserción. Los tres factores más determinantes son la asistencia a clases, el número de materias reprobadas y si el estudiante trabaja. Por ejemplo, para el estudiante X, su baja asistencia (65%) y su condición de trabajar son los principales factores que lo ubican en la zona de riesgo. Recomendamos implementar un sistema de alertas tempranas basado en asistencia y ofrecer horarios flexibles para estudiantes que trabajan."*

---

## ✏️ Ejercicios

Pon en práctica lo aprendido con estos ejercicios.

### Ejercicio 1: Desempeño laboral con SHAP

Usando el dataset `rrhh_nomina.csv`, entrena un modelo para predecir `evaluacion_desempeno` a partir de las variables `edad`, `antiguedad`, `salario_mensual_usd` y `ausencias_anuales`. Luego aplica SHAP para explicar qué factores impulsan el desempeño.

Pasos:
1. Cargar el dataset
2. Preparar features y target
3. Entrenar un RandomForestRegressor
4. Calcular valores SHAP con TreeExplainer
5. Crear un summary plot y un bar plot
6. Interpretar: ¿Qué factores impulsan un mejor desempeño?

In [ ]:
# ============================================================
# EJERCICIO 1: SHAP para desempeño laboral
# ============================================================

# Paso 1: Cargar datos
# Tu código aquí


# Paso 2: Preparar features y target
# features_rrhh = ['edad', 'antiguedad', 'salario_mensual_usd', 'ausencias_anuales']
# Tu código aquí


# Paso 3: Entrenar RandomForestRegressor
# Tu código aquí


# Paso 4: Calcular valores SHAP
# Tu código aquí


# Paso 5: Crear summary plot y bar plot
# Tu código aquí


# Paso 6: Interpretar en comentarios
# ¿Qué factores impulsan mejor desempeño?
# Tu interpretación aquí


### Ejercicio 2: Recomendación de política contra la deserción

Usando el modelo de riesgo de deserción que entrenamos en la sección 6, identifica los **3 factores más importantes** según SHAP y escribe un párrafo de recomendación de política académica.

Formato sugerido:
```
Con base en el análisis de [N] estudiantes, el modelo identifica que los tres
factores que más contribuyen al riesgo de deserción son:
1. [Factor 1]: [explicación y evidencia del SHAP]
2. [Factor 2]: [explicación y evidencia del SHAP]
3. [Factor 3]: [explicación y evidencia del SHAP]

Recomendación: [Propuesta concreta de intervención]
```

In [ ]:
# ============================================================
# EJERCICIO 2: Recomendación de política
# ============================================================

# Paso 1: Obtener los 3 factores más importantes según SHAP
# Pista: usar shap_values_clf y calcular la media absoluta por variable
# Tu código aquí


# Paso 2: Crear un bar plot con los 3 factores principales
# Tu código aquí


# Paso 3: Escribir la recomendación como texto en una variable
# recomendacion = """
# Con base en el análisis de ... estudiantes...
# """
# print(recomendacion)
# Tu código aquí


### Ejercicio 3: Waterfall para un empleado específico

Usando el dataset `rrhh_nomina.csv` y el modelo que entrenaste en el Ejercicio 1:

1. Selecciona un empleado específico del conjunto de prueba
2. Crea un waterfall plot de SHAP para ese empleado
3. Escribe una interpretación en lenguaje claro (como si se lo explicaras a un gerente de RRHH)

Ejemplo de interpretación:
> "El modelo predice una evaluación de 3.8 para este empleado. Su alta antigüedad (+15 años) contribuye positivamente (+0.4), mientras que sus ausencias frecuentes (4 al año) reducen la predicción (-0.3)."

In [ ]:
# ============================================================
# EJERCICIO 3: Waterfall para un empleado
# ============================================================

# Paso 1: Seleccionar un empleado del conjunto de prueba
# Tu código aquí


# Paso 2: Mostrar sus datos y la predicción del modelo
# Tu código aquí


# Paso 3: Crear waterfall plot
# Tu código aquí


# Paso 4: Escribir interpretación en lenguaje claro
# Tu código aquí


---

## 📋 Resumen

En esta sesión aprendimos a abrir la "caja negra" de los modelos de Machine Learning:

| Técnica | ¿Qué hace? | Ventaja principal |
|---------|-----------|-------------------|
| **Feature Importance** (nativa) | Mide cuánto usó cada variable el modelo internamente | Rápida y fácil de calcular |
| **Permutation Importance** | Mide cuánto empeora el modelo al desordenar cada variable | Más robusta, funciona con cualquier modelo |
| **SHAP Summary Plot** | Muestra importancia global + dirección del efecto | Visión completa de todas las variables |
| **SHAP Bar Plot** | Ranking de importancia promedio | Simple y claro para presentaciones |
| **SHAP Waterfall** | Explica UNA predicción individual variable por variable | Ideal para casos específicos |
| **SHAP Dependence Plot** | Muestra cómo varía el efecto de una variable según su valor | Detecta relaciones no lineales |

### Flujo de trabajo recomendado:

```
Entrenar modelo → Feature Importance (vista rápida) → SHAP Summary (vista profunda) →
Waterfall (casos individuales) → Comunicar en lenguaje claro
```

### Mensaje clave:

**Un modelo que no se puede explicar es un modelo que no se puede usar para tomar decisiones responsables.** SHAP nos da el puente entre la complejidad del algoritmo y la claridad que necesitan los tomadores de decisiones.

## 📚 Referencias

1. **SHAP — Documentación oficial:**  
   https://shap.readthedocs.io/en/latest/

2. **Lundberg, S.M. & Lee, S.I. (2017). *A Unified Approach to Interpreting Model Predictions.* NeurIPS:**  
   https://papers.nips.cc/paper/7062-a-unified-approach-to-interpreting-model-predictions (Paper original de SHAP)

3. **Molnar, C. — *Interpretable Machine Learning* (libro gratuito en línea):**  
   https://christophm.github.io/interpretable-ml-book/ (Recurso excelente y accesible)

4. **scikit-learn — Permutation Importance:**  
   https://scikit-learn.org/stable/modules/permutation_importance.html

5. **Valores de Shapley — Explicación visual (en español):**  
   https://www.cienciadedatos.net/documentos/py52-shap-shapley-values-machine-learning.html